In [1]:
from typing import Tuple
import torch
import torch as T
import torch.nn as nn

class ChemVAE(nn.Module):

    def __init__(self, input_dim: int = 64, latent_dim: int = 4):

        super(ChemVAE, self).__init__()
        self._input_dim = input_dim
        self._latent_dim = latent_dim

        self.fc1 = nn.Linear(self._input_dim, 32)
        self.fc2a = nn.Linear(32, self._latent_dim)
        self.fc2b = nn.Linear(32, self._latent_dim)
        self.fc3 = nn.Linear(self._latent_dim, 32)
        self.fc4 = nn.Linear(32, self._input_dim)

    def encode(self, x: 'torch.tensor') -> Tuple['torch.tensor', 'torch.tensor']:

        z = T.sigmoid(self.fc1(x))
        mean = T.sigmoid(self.fc2a(z))
        logvar = T.sigmoid(self.fc2b(z))
        return (mean, logvar)

    def decode(self, z: 'torch.tensor') -> 'torch.tensor':

        z = T.sigmoid(self.fc3(z))
        z = T.sigmoid(self.fc4(z))
        return z

    @staticmethod
    def generate(mean: 'torch.tensor', logvar: 'torch.tensor') -> 'torch.tensor':

        stdev = T.exp(0.5 * logvar)
        noise = T.randn_like(stdev)
        return mean + (noise * stdev)

    def forward(self, x: 'torch.tensor') -> Tuple['torch.tensor', 'torch.tensor', 'torch.tensor']:

        mean, logvar = self.encode(x)
        inpt = self.generate(mean, logvar)
        recon_x = self.decode(inpt)
        return (recon_x, mean, logvar)


def elbo_loss(recon_x: 'torch.tensor', x: 'torch.tensor', mean: 'torch.tensor', logvar: 'torch.tensor', beta: float = 1.0):
    # https://arxiv.org/abs/1312.6114

    bce = T.nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * T.sum(1 + logvar - T.pow(mean, 2) - T.exp(logvar))
    return bce + (beta * kld)

In [2]:
from sklearn.model_selection import train_test_split

nd_1 = torch.distributions.normal.Normal(-4, 1)  # N(0, 1)
nd_2 = torch.distributions.normal.Normal(4, 1)  # N(4, 1)

data_1 = nd_1.sample((16, 64))  # 10 samples of 64
data_2 = nd_2.sample((16, 64))  # 10 samples of 64

data_1_train, data_1_test, data_2_train, data_2_test = train_test_split(data_1, data_2, test_size=0.2)

training_set = T.cat((data_1_train, data_2_train))
testing_set = T.cat((data_1_test, data_2_test))

vae = ChemVAE()
vae.train()
opt = T.optim.Adam(vae.parameters(), lr=0.001)

for ep in range(500):

    opt.zero_grad()
    recon_x, mean, logvar = vae(training_set)
    loss_val = elbo_loss(recon_x, training_set, mean, logvar, 0.1)
    loss_val.backward()
    epoch_loss = loss_val.item()
    opt.step()

    if ep % 5 == 0:
        print(f'{ep} | {epoch_loss}')

0 | 1083.3961181640625
5 | 987.213134765625
10 | 931.0475463867188
15 | 856.1986083984375
20 | 812.8037719726562
25 | 753.1317138671875
30 | 584.2843627929688
35 | 546.1512451171875
40 | 510.38958740234375
45 | 433.5334167480469
50 | 320.91015625
55 | 438.20849609375
60 | 160.2130889892578
65 | 182.68807983398438
70 | -60.92292785644531
75 | -385.3905029296875
80 | -51.140106201171875
85 | -731.3997192382812
90 | -662.5484008789062
95 | -770.4038696289062
100 | -420.1897277832031
105 | -1055.4775390625
110 | -734.8348999023438
115 | -1187.2879638671875
120 | -1565.833984375
125 | -1861.1727294921875
130 | -2097.64013671875
135 | -847.97802734375
140 | -2381.45166015625
145 | -2403.158447265625
150 | -1204.72900390625
155 | -2053.302001953125
160 | -3872.89794921875
165 | -2731.099853515625
170 | -3252.473876953125
175 | -2739.287353515625
180 | -5708.7529296875
185 | -3314.86572265625
190 | -3686.679931640625
195 | -4102.6650390625
200 | -3456.58740234375
205 | -6132.4248046875
210 | -

In [3]:
pred_test = vae(testing_set)
print(pred_test[0].detach().numpy())

[[1.62581571e-07 1.09121508e-07 3.28188008e-07 8.47516333e-08
  1.34467228e-07 5.99263288e-08 1.21242124e-07 1.05480339e-07
  1.18137109e-07 1.19969002e-07 1.00369007e-07 4.89229421e-08
  1.57729062e-07 1.74963635e-08 1.88536958e-07 5.93938907e-08
  5.16410061e-08 5.03449257e-07 3.51211256e-08 2.88671576e-08
  1.44664503e-08 1.91954697e-07 2.12965190e-07 7.09478343e-08
  5.02216295e-08 1.73526800e-07 2.02121385e-07 3.12125366e-08
  9.26595192e-08 2.51352930e-08 3.01616012e-08 2.90817450e-08
  3.72510733e-08 1.52242194e-07 6.09027779e-08 9.85783686e-08
  4.32737757e-08 2.60558409e-07 1.00389492e-07 5.00524635e-08
  1.34118494e-07 1.86012500e-07 9.16051661e-08 5.56076678e-08
  1.45025382e-07 1.77316721e-07 8.59309282e-08 5.08446512e-08
  5.00005626e-07 6.74810252e-08 4.03773122e-08 7.10213541e-08
  5.68217970e-08 4.61735929e-08 8.84581652e-08 5.26059303e-08
  2.51230219e-08 4.50216078e-07 3.01326217e-08 1.81375270e-07
  3.38474955e-08 1.50611953e-07 3.77052736e-08 3.36291528e-08]
 [4.194